In [4]:
import requests
import pandas as pd
from numpy import percentile

#Función para leer los datos de un JSON mediante un enlace
def leer_json(url):
  read=requests.get(url)
  return read.json()

In [5]:

#Función para extraer información precisa de la BD
# ID del CVE, Puntaje CVSS, Fecha de publicación
def extraer_info(lista_datos):
  archivero=[]
  #Recorrer cada diccionario de la lista
  for dicc in lista_datos:
    archivo=[]
    #Extraer información
    #CVE
    cve=dicc["CVE"]
    archivo.append(cve)
    #Puntaje CVSS
    cvss3_score=dicc.get("cvss3_score")
    #Convertir str a float
    if cvss3_score is not None:
      try:
        cvss3_score=float(cvss3_score)
        if cvss3_score ==0.0:
          archivo.append("None")
        elif 0.1<= cvss3_score <= 3.9:
          archivo.append("Low")
        elif 4.0 <= cvss3_score <= 6.9:
          archivo.append("Medium")
        elif 7.0 <= cvss3_score <= 8.9:
          archivo.append("High")
        elif cvss3_score >= 9.0:
          archivo.append("Critical")
      except ValueError:
        archivo.append("CVSS3 no encontrado")
    archivo.append(cvss3_score)
    #Severity
    severity=dicc["severity"]
    archivo.append(severity)
    #public date
    public_date=dicc["public_date"]
    archivo.append(public_date)

    #Guardar datos encontrados en el archivero
    archivero.append(archivo)
  return archivero

#Función para convertir la lista del archivero a un archivo CSV
def convertir_csv(archivero):
  df1=pd.DataFrame(archivero,columns=["CVE","CVSS","CVSS Puntaje","Severity","Public Date"])

  #Guardar CSV
  df1.to_csv("CSV_Tabla1.csv",index=False)



#EJECUCIÓN DEL PROGRAMA
url= "https://access.redhat.com/hydra/rest/securitydata/cve.json"
lista_datos=leer_json(url)
archivero=extraer_info(lista_datos)
convertir_csv(archivero)





In [7]:
#Función para extraer información precisa de la BD
# ID del CVE,Productos
def extraer_info(lista_datos):
  archivero2=[]
  #Recorrer cada diccionario de la lista
  for dicc in lista_datos:
    #CVE
    cve=dicc["CVE"]
    #Función para obtener URL con el detalle de resources
    #url de resources
    resource_url=dicc.get("resource_url")
    #Si no hay URL, continuar
    if not resource_url:
      continue
    #Crear lista con los recursos encontrados
    productos_encontrados=leer_json(resource_url)
    #verificar que no este vacía para analizar el contenido
    if not productos_encontrados:
      print(f"Recursos no encontrados en la URL del CVE:{cve}")
    else:
        #Revisar que el package_state no este vacío
        package=productos_encontrados.get("package_state")
        #Solo si existe el package, buscamos productos
        if package:
          #Buscar cada producto
          for prod in package:
            productos=prod.get("product_name")

            #Crear dupla CVE, Productos. Como si fuera nuestro archivo
            fila=[cve,productos]
            #Almacenar cada dupla en el archivero
            archivero2.append(fila)
        else:
          fila = [cve, "Producto no especificado"]
          archivero2.append(fila)


  return archivero2

#Función para convertir la lista del archivero a un archivo CSV
def convertir_csv(archivero2):
  df2=pd.DataFrame(archivero2,columns=["CVE","Productos"])

  #Guardar CSV
  df2.to_csv("CSV_Tabla2.csv",index=False)



#EJECUCIÓN DEL PROGRAMA
url= "https://access.redhat.com/hydra/rest/securitydata/cve.json"
lista_datos=leer_json(url)
archivero=extraer_info(lista_datos)
convertir_csv(archivero)





In [9]:
#Función para conectarse a la BD del CISA KEV
def extraer_info(lista_datos):
  archivero3=[]
  #Buscar la lista de vulnerabilidades
  vulnerabilities=lista_datos.get("vulnerabilities")
  if vulnerabilities:
    #Recorrer cada diccionario de la lista
    for dicc in vulnerabilities:
      #CVE
      cve_cisa=dicc["cveID"]
      archivero3.append(cve_cisa)
    cves_cisa_set = set(archivero3)

  return cves_cisa_set

#Función para comparar el CVE obtenido del CISA KEV con el csv de la tabla 2
def comparar_csv(cves_cisa_set):
  #Leer el archivo de tabla2 para encontrar CVE y Productos
  df2=pd.read_csv("CSV_Tabla2.csv")
  df2.loc[len(df2)] = ["CVE-2023-24998", "Producto de Prueba para Validación"]
  lista_final=[]
  #Recorrer la  con iteraciones
  for i, fila_df2 in df2.iterrows():
    cve=fila_df2["CVE"]
    productos=fila_df2["Productos"]

    #Existe este CVE en el catalogo CISA?
    if cve in cves_cisa_set:
      #Mandar a llamar la API de EPSS para buscar en el catalogo por CVE
      url_EPSS=f"https://api.first.org/data/v1/epss?cve={cve}"
      #Extraer los datos encontrados
      datos_EPSS=leer_json(url_EPSS)

      #Si no es lista vacía
      if datos_EPSS:
        #Buscar en el diccionario la data
        lista_EPSS=datos_EPSS.get("data")
        if lista_EPSS:
          ##Buscar en la posición 0 de la lista para sacar el diccionario
          diccionario_EPSS=lista_EPSS[0]

          #Extraer información del diccionario EPSS
          epss=diccionario_EPSS["epss"]
          percentile=diccionario_EPSS["percentile"]

          #Armar la fila de datos
          fila=[cve,productos,epss,percentile]
          lista_final.append(fila)


  return lista_final
#Función para convertir la lista del archivero a un archivo CSV
def convertir_csv(lista_final):
  df3=pd.DataFrame(lista_final,columns=["CVE","Productos","EPSS","Percentile"])

  #Guardar CSV
  df3.to_csv("CSV_Tabla3.csv",index=False)




#EJECUCIÓN DEL PROGRAMA
url= "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"
lista_datos=leer_json(url)
archivero=extraer_info(lista_datos)
archivero3=comparar_csv(archivero)
convertir_csv(archivero3)